# Session 27: Probability and Simulation

Data Science with Python, Level 2, Stage 4: Statistics

## Learning Objectives

By the end of this session you will be able to:

1. Express probabilities as numbers from 0 to 1.
2. Use the complement, and and or rules.
3. Estimate probabilities by simulation.
4. See the law of large numbers in action.
5. Calculate conditional probabilities from data.
6. Use the binomial distribution.

## Where We Are

In Session 26 you described data you already had. Statistics also asks: how likely is a result **by chance**? Was a new website design really better, or just lucky? Is a 2 percent fraud rate normal, or a warning?

**Probability** measures how likely an event is, from 0 (impossible) to 1 (certain). Some probabilities can be calculated with a formula. Many cannot, but a computer can **simulate** the situation thousands of times and count. With NumPy, a million experiments take less than a second.

## Setup

Run this cell first. `rng` is the seeded random generator from Session 15, so your results match these notes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(27)
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"

## 1. What Is a Probability?

In [ ]:
rolls = rng.integers(1, 7, size=12)
print(rolls)
print("Share of sixes:", (rolls == 6).mean())

many = rng.integers(1, 7, size=100_000)
print("Share of sixes in 100,000 rolls:", (many == 6).mean())
print("Exact:", round(1 / 6, 5))

## 2. Rules of Probability

| Rule | Formula | Example |
| --- | --- | --- |
| Complement (not) | P(not A) = 1 - P(A) | P(no six) = 1 - 1/6 = 5/6 |
| And, independent events | P(A and B) = P(A) x P(B) | Two sixes in two rolls: 1/36 |
| Or, events that cannot both happen | P(A or B) = P(A) + P(B) | A 1 or a 2: 2/6 |
| At least one | 1 - P(none) | At least one six in 4 rolls: 1 - (5/6)^4 |

In [ ]:
exact = 1 - (5 / 6) ** 4

games = rng.integers(1, 7, size=(100_000, 4))
simulated = (games == 6).any(axis=1).mean()

print("Exact:    ", round(exact, 4))
print("Simulated:", round(simulated, 4))

## 3. Simulation

In [ ]:
def shared_birthday(people, trials=20_000):
    days = rng.integers(1, 366, size=(trials, people))
    shared = [len(set(row)) < people for row in days]
    return np.mean(shared)

for people in [10, 23, 30, 50]:
    print(people, "people:", round(shared_birthday(people), 3))

In [ ]:
n_days = 10_000
customers = rng.poisson(lam=40, size=n_days)
spend = customers * rng.normal(6.5, 1.2, size=n_days)

print("Average daily revenue:", round(spend.mean(), 1))
print("Chance of a day below 200:", round((spend < 200).mean(), 3))

sns.histplot(spend, bins=40)
plt.title("Simulated daily revenue of a cafe")
plt.show()

## 4. The Law of Large Numbers

In [ ]:
flips = rng.integers(0, 2, size=5000)
running_share = flips.cumsum() / np.arange(1, 5001)

plt.plot(running_share)
plt.axhline(0.5, color="tab:red", linestyle="--")
plt.title("Share of heads as the number of flips grows")
plt.xlabel("Number of flips")
plt.ylabel("Share of heads")
plt.show()

## 5. Conditional Probability

A **conditional probability** P(A given B) is the probability of A among only the cases where B happened. With data, it is a share inside a group, which is exactly what `groupby` and `crosstab` calculate. Here we use a practice dataset of 2,000 customers of a fictional phone company, some of whom cancelled their contract, or **churned**.

In [ ]:
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = churn["churn"] == "Yes"

print("P(left):", round(churn["left"].mean(), 3))
print(churn.groupby("contract")["left"].mean().round(3))

In [ ]:
pd.crosstab(churn["contract"], churn["left"], normalize="index").round(3)

## 6. The Binomial Distribution

When the same yes or no experiment is repeated `n` times with the same probability `p`, the number of successes follows a **binomial distribution**. Examples: the number of sixes in 10 rolls, the number of customers out of 20 who buy, the number of faulty items in a box of 50.

In [ ]:
n, p = 20, 0.3
k = np.arange(0, n + 1)
probs = stats.binom.pmf(k, n, p)

plt.bar(k, probs)
plt.title("Customers who buy, out of 20, when p = 0.3")
plt.xlabel("Number who buy")
plt.ylabel("Probability")
plt.show()
print("P(exactly 6):", round(stats.binom.pmf(6, n, p), 3))
print("P(10 or more):", round(1 - stats.binom.cdf(9, n, p), 3))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Wrong answer | Adding probabilities of events that can both happen | Subtract the overlap, or simulate. |
| Gambler's fallacy | Thinking a coin is due for heads after five tails | Independent flips have no memory. |
| Confused direction | Treating P(A given B) as P(B given A) | Say carefully which group the share is taken in. |
| Noisy estimate | Trusting a simulation with very few trials | Use thousands of trials; check that results are stable. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Two Dice

Simulate 100,000 rolls of two dice and estimate the probability that the total is 7. Compare with the exact value 6/36.

In [ ]:
# Write your code below

### In-Class Exercise 2: Exam Guessing

A quiz has 10 questions with 4 options each. A student guesses every answer. Use `stats.binom` to find the chance of passing with 5 or more correct.

In [ ]:
# Write your code below

### In-Class Exercise 3: Fraud Rate

Load `card_transactions.csv`. Find P(fraud), P(fraud given online) and P(fraud given a new merchant). Which clue raises the risk most?

In [ ]:
# Write your code below

### In-Class Exercise 4: Sample Means

Draw 2,000 samples of 30 dice rolls each and compute the mean of every sample. Draw a histogram of the 2,000 means. What shape do you see?

In [ ]:
# Write your code below

## Summary

- A probability is a number from 0 (impossible) to 1 (certain).
- Complement: P(not A) = 1 - P(A); independent and: multiply.
- Simulation: generate many trials and count.
- More trials bring the share closer to the true probability.
- A conditional probability is a rate inside a group.
- The binomial counts successes in n independent tries.

## What's Next

The last in-class exercise showed something remarkable: sample means form a bell curve. Next session this becomes the most useful idea in statistics, and it lets us say how sure we are about an average.